In [0]:
# Databricks environment configuration
catalog_name = "learning_projects"
schema_name = "retail_project"

table_prefix = f"{catalog_name}.{schema_name}"

In [0]:
bronze_customers_df = spark.table(f"{table_prefix}.bronze_customers")
bronze_customers_df.printSchema()
print("Bronze customers count: ", bronze_customers_df.count())

In [0]:
from pyspark.sql.functions import upper, trim, col
silver_customer_df = (
    bronze_customers_df
    .withColumn("customer_city", upper(trim(col("customer_city"))))
    .withColumn("customer_state", upper(trim(col("customer_state"))))
)
silver_customer_df.show(10, truncate= False)

In [0]:
from pyspark.sql.functions import col, length

# Define valid customer conditions
valid_customer_condition = (
    col("customer_id").isNotNull()
    & col("customer_unique_id").isNotNull()
    & col("customer_city").isNotNull()
    & (length(col("customer_state")) == 2).eqNullSafe(True)
)

# Valid customers
valid_customers_df = silver_customer_df.filter(
    valid_customer_condition
)

# Rejected customers
rejected_customers_df = silver_customer_df.filter(
    ~valid_customer_condition
)

# Validate record counts
total_count = silver_customer_df.count()
valid_count = valid_customers_df.count()
rejected_count = rejected_customers_df.count()

print("Total customers:", total_count)
print("Valid customers:", valid_count)
print("Rejected customers:", rejected_count)
print("Reconciliation:", total_count == valid_count + rejected_count)

In [0]:
from pyspark.sql.functions import count, col
duplicate_customers_df = (
    valid_customers_df
    .groupBy("customer_id")
    .agg(
        count("*").alias("count")
    )
    .filter(col("count") > 1)
)
duplicate_customers_df.show()

In [0]:
valid_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{table_prefix}.silver_customers")

In [0]:
spark.sql(f"""
    SELECT COUNT(*) AS customer_count
    FROM {table_prefix}.silver_customers
""").show()

In [0]:
bronze_products_df = (
    spark.table(f"{table_prefix}.bronze_products")
)

In [0]:
from pyspark.sql.functions import coalesce, col, lit, when

silver_products_df = (
    bronze_products_df
    .withColumnRenamed("product_name_lenght","product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")
    .withColumn("product_category_name", coalesce(col("product_category_name"), lit("UNKNOWN")))
    .withColumn("product_weight_g",
                when (col("product_weight_g") <= 0, lit(None).cast("int")).otherwise(col("product_weight_g")))
)
#silver_products_df.show(10, truncate = False)
#silver_products_df.printSchema()
print(
    "UNKNOWN categories:",
    silver_products_df
    .filter(col("product_category_name") == "UNKNOWN")
    .count()
)

print(
    "Invalid weights:",
    silver_products_df
    .filter(col("product_weight_g") <= 0)
    .count()
)

print(
    "NULL weights:",
    silver_products_df
    .filter(col("product_weight_g").isNull())
    .count()
)

In [0]:
print("Null Product ID's",
      silver_products_df
      .filter(col("product_id").isNull())
      .count()
      )
duplicate_products_df = ( 
      silver_products_df
      .groupBy("product_id")
      .agg(count("*").alias("count"))
      .filter(col("count")> 1)
)
duplicate_products_df.show()
    

In [0]:
(
    silver_products_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{table_prefix}.silver_products")
)

In [0]:
spark.sql(f"""
    SELECT COUNT(*) AS customer_count
    FROM {table_prefix}.silver_products
""").show()

In [0]:
bronze_orders_df = spark.table(f"{table_prefix}.bronze_orders")

In [0]:
bronze_orders_df.show()

In [0]:
from pyspark.sql.functions import trim, lower

silver_orders_df = (
    bronze_orders_df
    .withColumn("order_status", trim(lower(col("order_status")))
                )
)

In [0]:
from pyspark.sql.functions import col
(
    silver_orders_df
    .groupBy("order_status")
    .count()
    .orderBy(col("count").desc())
    .show()
)

In [0]:
valid_orders_df = (
    silver_orders_df
    .filter(
        (col("order_id").isNotNull()) &
        (col("customer_id").isNotNull()) &
        (col("order_purchase_timestamp").isNotNull())
    )
)

rejected_orders_df = (
    silver_orders_df
    .filter(~(
        (col("order_id").isNotNull()) &
        (col("customer_id").isNotNull()) &
        (col("order_purchase_timestamp").isNotNull())
    ))
)
print("silver orders: ", silver_orders_df.count())
print("Valid orders:", valid_orders_df.count())
print("Rejected orders:", rejected_orders_df.count())

In [0]:
invalid_approval_df = (
    valid_orders_df
    .filter(col("order_approved_at") < col("order_purchase_timestamp"))
)
print("invalid approval chronology: ", invalid_approval_df.count())

In [0]:
invalid_delivery_sequence_df = (
    valid_orders_df
    .filter(col("order_delivered_customer_date") < col("order_delivered_carrier_Date"))
)
print("invalid delivery sequence",
      invalid_delivery_sequence_df.count()
      )

In [0]:
valid_orders_df = (
    valid_orders_df
    .withColumn("has_timestamp_issue",
                when(  (col("order_approved_at") < col("order_purchase_timestamp"))
                | (col("order_delivered_customer_date") < col("order_delivered_carrier_Date"))
                |((col("order_status") == "delivered")
                  & (col("order_approved_at").isNull() |
                     col("order_delivered_carrier_Date").isNull() |
                     col("order_delivered_customer_date").isNull()
                     )
                  ), 
                True 
                ).otherwise(False)
                )
    .withColumn("is_late_delivery",
                when(
                    (col("order_status") == "delivered")
                    & (col("order_delivered_customer_date") > col("order_estimated_delivery_date")),
                    True
                ).otherwise(False))
)
valid_orders_df.groupBy("has_timestamp_issue").count().show()
#valid_orders_df.groupBy("is_late_delivery").count().show()

In [0]:
delivered_missing_timestamp_df = (
    valid_orders_df
    .filter((col("order_status") == "delivered")
            &
            (
                (col("order_approved_at").isNull() )
                | (col("order_delivered_carrier_Date").isNull())
                | (col("order_delivered_customer_date").isNull())
            ))
)
print(
    "Delivered orders with missing timestamps:",
    delivered_missing_timestamp_df.count()
)

In [0]:
print("null order_id:",
    valid_orders_df
    .filter(col("order_id").isNull())
    .count()
)
duplicate_orders_df = (
    valid_orders_df
    .groupBy("order_id")
    .agg(count("*").alias("count"))
    .filter(col("count")> 1)
)
duplicate_orders_df.show()

In [0]:
(
    valid_orders_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{table_prefix}.silver_orders")
)

In [0]:
spark.sql(f"""
    SELECT COUNT(*) AS customer_count
    FROM {table_prefix}.silver_orders
""").show()

In [0]:
bronze_order_items_df = spark.table(f"{table_prefix}.bronze_order_items")

In [0]:
from pyspark.sql.functions import col, count
(
    bronze_order_items_df
    .groupBy("order_id", "order_item_id")
    .agg(count("*").alias("count"))
    .filter(col("count") > 1)
    .show(100)
)
print("null order id :",
      bronze_order_items_df
      .filter(col("order_id").isNull())
      .count()
)
print("null order item_id: ",
      bronze_order_items_df
      .filter(col("order_item_id").isNull())
      .count()
      )

In [0]:
from pyspark.sql.functions import col, sum, when
(
    bronze_order_items_df
    .agg(
        sum(when(col("price").isNull(), 1).otherwise(0)).alias("null_price"),
        sum(when(col("price") <= 0, 1).otherwise(0)).alias("invalid_price"),
        sum(when(col("freight_value").isNull(), 1).otherwise(0)).alias("null_freight"),
        sum(when(col("freight_value")<0, 1).otherwise(0)).alias("invalid_freight")
        )
    .show()
)

In [0]:

orphan_order_items_df = (
    bronze_order_items_df.alias("oi").
    join(
        valid_orders_df.alias("o"),
        col("oi.order_id") == col("o.order_id"), "left_anti" )
)
print("orphan order item: ", orphan_order_items_df.count())

In [0]:
orphan_products_df = (
    bronze_order_items_df.alias("oi").join(
        bronze_products_df.alias("p"), col("oi.product_id") == col("p.product_id"), "left_anti"
)
)
print("orphan_products_df :", orphan_products_df.count())

In [0]:
(
    bronze_order_items_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{table_prefix}.silver_order_items")
)

In [0]:
spark.sql(f"""
    SELECT COUNT(*) AS customer_count
    FROM {table_prefix}.silver_order_items
""").show()